In [1]:
!pip install -q datasets

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/indic_langid'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/indic_langid


In [ ]:
import re

_SENT_END_CHARS = ".!?।॥۔؟"
_MULTI_PUNCT_RE = re.compile(rf"([{_SENT_END_CHARS}])\1+")
_SPLIT_RE = re.compile(rf"(?<=[{_SENT_END_CHARS}])\s+")
_WS_RE = re.compile(r"\s+")


def sentence_tokenize(paragraph: str):
    if not paragraph:
        return []
    text = paragraph.replace("\n", " ").replace("\t", " ").strip() #remove newlines and tabs
    text = _WS_RE.sub(" ", text) # remove space
    text = _MULTI_PUNCT_RE.sub(r"\1", text) # remove duplicate punctuation
    if not text:
        return []
    pieces = _SPLIT_RE.split(text)
    return [p.strip() for p in pieces if p.strip()]

In [ ]:
import json
import random
from pathlib import Path
from datasets import load_dataset

SEED = 42
REPO_ID = "ai4bharat/IndicCorpV2"

LANGUAGES = [
    "asm_Beng", "ben_Beng", "brx_Deva", "doi_Deva", "gom_Deva", "guj_Gujr",
    "hin_Deva", "kan_Knda", "kas_Arab", "mai_Deva", "mal_Mlym", "mar_Deva",
    "mni_Mtei", "npi_Deva", "ory_Orya", "pan_Guru", "san_Deva", "snd_Deva",
    "tam_Taml", "tel_Telu", "urd_Arab", "khasi", "santhali",
]

SENTS_PER_LANG = 1000
MIN_CHARS = 15
MIN_WORDS = 3
MAX_PARAGRAPHS_TO_SCAN = 200_000
OUT_DIR = Path("data")

random.seed(SEED)


def build_language_dataset():
    OUT_DIR.mkdir(exist_ok=True)
    print(f"Using {len(LANGUAGES)} languages:\n{LANGUAGES}\n")
    all_rows = [] #tuples 

    for lang in LANGUAGES:
        out_path = OUT_DIR / f"_raw_{lang}.jsonl"
        if out_path.exists():
            print(f"'{lang}' already collected, skipping (delete {out_path} to redo)")
            with open(out_path, encoding="utf-8") as f:
                for line in f:
                    row = json.loads(line)
                    all_rows.append((row["text"], row["label"]))
            continue

        print(f"=== Collecting sentences for '{lang}' ===", flush=True)
        collected = set()
        ds = load_dataset(REPO_ID, split=lang, streaming=True)

        scanned = 0
        for example in ds:
            scanned += 1
            if scanned % 2000 == 0:
                print(f"    scanned {scanned}, collected {len(collected)}...", flush=True)
            para = example.get("text", "") or example.get("sentence", "")
            if not para:
                continue

            for sent in sentence_tokenize(para):
                if len(sent) < MIN_CHARS or len(sent.split()) < MIN_WORDS:
                    continue
                if sent in collected:
                    continue
                collected.add(sent)
                if len(collected) >= SENTS_PER_LANG:
                    break

            if len(collected) >= SENTS_PER_LANG or scanned >= MAX_PARAGRAPHS_TO_SCAN:
                break

        print(f"  -> collected {len(collected)} sentences (scanned {scanned} paragraphs)")
        if len(collected) < SENTS_PER_LANG:
            print(f"  WARNING: only found {len(collected)}/{SENTS_PER_LANG} for '{lang}'")

        # save per-language raw file immediately, so a crash doesn't lose earlier languages
        with open(out_path, "w", encoding="utf-8") as f:
            for s in collected:
                f.write(json.dumps({"text": s, "label": lang}, ensure_ascii=False) + "\n")

        for s in collected:
            all_rows.append((s, lang))

    return all_rows


def stratified_split(rows, train_ratio=0.8, val_ratio=0.1):
    by_label = {}
    for text, label in rows:
        by_label.setdefault(label, []).append(text)

    train, val, test = [], [], []
    for label, texts in by_label.items():
        texts = texts[:]
        random.shuffle(texts)
        n = len(texts)
        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))
        train += [(t, label) for t in texts[:n_train]]
        val += [(t, label) for t in texts[n_train:n_train + n_val]]
        test += [(t, label) for t in texts[n_train + n_val:]]

    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)
    return train, val, test


def save_split(rows, name):
    path = OUT_DIR / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for text, label in rows:
            f.write(json.dumps({"text": text, "label": label}, ensure_ascii=False) + "\n")
    print(f"Saved {len(rows)} rows -> {path}")


rows = build_language_dataset()
train, val, test = stratified_split(rows)
save_split(train, "train")
save_split(val, "val")
save_split(test, "test")
print("\nDone.")

Using 23 languages:
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab', 'khasi', 'santhali']

=== Collecting sentences for 'asm_Beng' ===


README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

  -> collected 1000 sentences (scanned 951 paragraphs)
=== Collecting sentences for 'ben_Beng' ===
  -> collected 1000 sentences (scanned 585 paragraphs)
=== Collecting sentences for 'brx_Deva' ===
  -> collected 1000 sentences (scanned 1211 paragraphs)
=== Collecting sentences for 'doi_Deva' ===
  -> collected 1000 sentences (scanned 1531 paragraphs)
=== Collecting sentences for 'gom_Deva' ===
  -> collected 1000 sentences (scanned 993 paragraphs)
=== Collecting sentences for 'guj_Gujr' ===
  -> collected 1000 sentences (scanned 741 paragraphs)
=== Collecting sentences for 'hin_Deva' ===
  -> collected 1000 sentences (scanned 627 paragraphs)
=== Collecting sentences for 'kan_Knda' ===
  -> collected 1000 sentences (scanned 887 paragraphs)
=== Collecting sentences for 'kas_Arab' ===
  -> collected 1000 sentences (scanned 1371 paragraphs)
=== Collecting sentences for 'mai_Deva' ===
  -> collected 1000 sentences (scanned 965 paragraphs)
=== Collecting sentences for 'mal_Mlym' ===
  -> co

In [7]:
# tfidf
import math
import re
from collections import Counter, defaultdict

_WORD_RE = re.compile(r"\S+")


def _words(text):
    return _WORD_RE.findall(text)


def _word_ngrams(words, n):
    if len(words) < n:
        return []
    return ["_".join(words[i:i + n]) for i in range(len(words) - n + 1)]


def _char_ngrams(text, n):
    text = text.replace(" ", "")
    if len(text) < n:
        return []
    return [text[i:i + n] for i in range(len(text) - n + 1)]


def extract_features(text):
    text = text.strip().lower()
    words = _words(text)
    feats = []
    feats += ["W1|" + w for w in _word_ngrams(words, 1)]
    feats += ["W2|" + w for w in _word_ngrams(words, 2)]
    for n in (2, 3, 4):
        feats += [f"C{n}|" + g for g in _char_ngrams(text, n)]
    return feats


class TfidfVectorizer:
    def __init__(self, min_df=2, max_features=20000):
        self.min_df = min_df
        self.max_features = max_features
        self.vocab_ = {}
        self.idf_ = []

    def fit(self, corpus):
        df = defaultdict(int)
        for text in corpus:
            for f in set(extract_features(text)):
                df[f] += 1

        candidates = [(f, c) for f, c in df.items() if c >= self.min_df]
        candidates.sort(key=lambda x: x[1], reverse=True)
        if self.max_features is not None:
            candidates = candidates[: self.max_features]

        self.vocab_ = {f: i for i, (f, _) in enumerate(candidates)}

        n_docs = len(corpus)
        self.idf_ = [0.0] * len(self.vocab_)
        for f, i in self.vocab_.items():
            self.idf_[i] = math.log((1 + n_docs) / (1 + df[f])) + 1.0
        return self

    def transform(self, corpus):
        vectors = []
        for text in corpus:
            counts = Counter(extract_features(text))
            vec = {}
            for f, c in counts.items():
                j = self.vocab_.get(f)
                if j is None:
                    continue
                vec[j] = c * self.idf_[j]

            norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
            for j in vec:
                vec[j] /= norm

            vectors.append(vec)
        return vectors

    def fit_transform(self, corpus):
        self.fit(corpus)
        return self.transform(corpus)

    @property
    def n_features(self):
        return len(self.vocab_)

In [8]:
#logistic regression
import math
import random


class LogisticRegression:
    def __init__(self, n_features, n_classes, lr=0.5, epochs=20,
                 batch_size=64, l2=1e-4, verbose=True, seed=0):
        self.n_features = n_features
        self.n_classes = n_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2
        self.verbose = verbose
        random.seed(seed)
        self.W = [[0.0] * n_classes for _ in range(n_features)]
        self.b = [0.0] * n_classes

    def _logits(self, sparse_vec):
        logits = self.b[:]
        for idx, val in sparse_vec.items():
            row = self.W[idx]
            for k in range(self.n_classes):
                logits[k] += val * row[k]
        return logits

    @staticmethod
    def _softmax(logits):
        m = max(logits)
        exps = [math.exp(z - m) for z in logits]
        s = sum(exps)
        return [e / s for e in exps]

    def _one_hot(self, label):
        y = [0.0] * self.n_classes
        y[label] = 1.0
        return y

    def fit(self, X, y, X_val=None, y_val=None):
        n = len(X)
        indices = list(range(n))

        for epoch in range(self.epochs):
            random.shuffle(indices)
            total_loss = 0.0

            for start in range(0, n, self.batch_size):
                batch = indices[start:start + self.batch_size]
                bsz = len(batch)

                grad_W_updates = {}
                grad_b = [0.0] * self.n_classes

                for i in batch:
                    sparse_vec = X[i]
                    label = y[i]

                    logits = self._logits(sparse_vec)
                    probs = self._softmax(logits)
                    target = self._one_hot(label)

                    eps = 1e-12
                    total_loss += -math.log(probs[label] + eps)

                    diff = [probs[k] - target[k] for k in range(self.n_classes)]

                    for k in range(self.n_classes):
                        grad_b[k] += diff[k]

                    for idx, val in sparse_vec.items():
                        if idx not in grad_W_updates:
                            grad_W_updates[idx] = [0.0] * self.n_classes
                        row_grad = grad_W_updates[idx]
                        for k in range(self.n_classes):
                            row_grad[k] += val * diff[k]

                for idx, row_grad in grad_W_updates.items():
                    row = self.W[idx]
                    for k in range(self.n_classes):
                        grad = row_grad[k] / bsz + self.l2 * row[k]
                        row[k] -= self.lr * grad

                for k in range(self.n_classes):
                    self.b[k] -= self.lr * (grad_b[k] / bsz)

            if self.verbose:
                msg = f"epoch {epoch + 1:>3}/{self.epochs}  train_loss={total_loss / n:.4f}"
                if X_val is not None:
                    preds = self.predict(X_val)
                    acc = sum(1 for p, t in zip(preds, y_val) if p == t) / len(y_val)
                    msg += f"  val_acc={acc:.4f}"
                print(msg)
        return self

    def predict_proba(self, X):
        return [self._softmax(self._logits(x)) for x in X]

    def predict(self, X):
        preds = []
        for x in X:
            logits = self._logits(x)
            best_k = max(range(self.n_classes), key=lambda k: logits[k])
            preds.append(best_k)
        return preds

In [9]:
#metrics
def confusion_counts(y_true, y_pred, n_classes):
    tp = [0] * n_classes
    fp = [0] * n_classes
    fn = [0] * n_classes
    for t, p in zip(y_true, y_pred):
        if t == p:
            tp[t] += 1
        else:
            fp[p] += 1
            fn[t] += 1
    return tp, fp, fn


def precision_recall_f1_per_class(y_true, y_pred, n_classes):
    tp, fp, fn = confusion_counts(y_true, y_pred, n_classes)

    precision = [0.0] * n_classes
    recall = [0.0] * n_classes
    f1 = [0.0] * n_classes

    for k in range(n_classes):
        denom_p = tp[k] + fp[k]
        denom_r = tp[k] + fn[k]
        precision[k] = tp[k] / denom_p if denom_p > 0 else 0.0
        recall[k] = tp[k] / denom_r if denom_r > 0 else 0.0
        denom_f = precision[k] + recall[k]
        f1[k] = 2 * precision[k] * recall[k] / denom_f if denom_f > 0 else 0.0

    return precision, recall, f1


def macro_f1(y_true, y_pred, n_classes):
    _, _, f1 = precision_recall_f1_per_class(y_true, y_pred, n_classes)
    return sum(f1) / len(f1)


def accuracy(y_true, y_pred):
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

In [12]:
#train
import json

def load_jsonl(path):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            texts.append(row["text"])
            labels.append(row["label"])
    return texts, labels


def main():
    print("Loading data...")
    train_texts, train_labels = load_jsonl("data/train.jsonl")
    val_texts, val_labels = load_jsonl("data/val.jsonl")
    test_texts, test_labels = load_jsonl("data/test.jsonl")

    label_set = sorted(set(train_labels))
    label2idx = {l: i for i, l in enumerate(label_set)}
    idx2label = {i: l for l, i in label2idx.items()}
    print(f"{len(label_set)} languages: {label_set}")

    y_train = [label2idx[l] for l in train_labels]
    y_val = [label2idx[l] for l in val_labels]
    y_test = [label2idx[l] for l in test_labels]

    print("\nFitting custom TF-IDF vectorizer (word 1/2-grams + char 2/3/4-grams)...")
    vectorizer = TfidfVectorizer(min_df=2, max_features=20000)
    X_train = vectorizer.fit_transform(train_texts)
    X_val = vectorizer.transform(val_texts)
    X_test = vectorizer.transform(test_texts)
    print(f"Vocabulary size: {vectorizer.n_features}")

    sample_idx = 0
    sample_text = train_texts[sample_idx]
    sample_label = train_labels[sample_idx]
    sample_vec = X_train[sample_idx]   # already computed above, no need to re-transform

    idx_to_feat = {i: f for f, i in vectorizer.vocab_.items()}

    print(f"\n--- TF-IDF example ---")
    print(f"Sentence: {sample_text}")
    print(f"Label: {sample_label}")
    print(f"Number of nonzero TF-IDF features: {len(sample_vec)}")
    print("Top 10 features by TF-IDF weight:")
    top_feats = sorted(sample_vec.items(), key=lambda x: x[1], reverse=True)[:10]
    for idx, weight in top_feats:
        print(f"  {idx_to_feat[idx]:<20} {weight:.4f}")
    print("--- end example ---\n")

    print("\nTraining custom softmax logistic regression...")
    clf = LogisticRegression(
        n_features=vectorizer.n_features,
        n_classes=len(label_set),
        lr=0.5,
        epochs=20,
        batch_size=64,
        l2=1e-4,
    )
    clf.fit(X_train, y_train, X_val, y_val)

    print("\nEvaluating on test set...")
    y_pred = clf.predict(X_test)
    macro = macro_f1(y_test, y_pred, len(label_set))
    acc = accuracy(y_test, y_pred)
    print(f"Test accuracy : {acc:.4f}")
    print(f"Test macro-F1 : {macro:.4f}")

    precision, recall, f1 = precision_recall_f1_per_class(y_test, y_pred, len(label_set))
    print(f"\n{'Language':<15}{'Precision':>10}{'Recall':>10}{'F1':>10}")
    for i, lang in idx2label.items():
        print(f"{lang:<15}{precision[i]:>10.3f}{recall[i]:>10.3f}{f1[i]:>10.3f}")


if __name__ == "__main__":
    main()

Loading data...
23 languages: ['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'khasi', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'santhali', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']

Fitting custom TF-IDF vectorizer (word 1/2-grams + char 2/3/4-grams)...
Vocabulary size: 20000

--- TF-IDF example ---
Sentence: नहि लिखय छथिन्ह.
Label: mai_Deva
Number of nonzero TF-IDF features: 18
Top 10 features by TF-IDF weight:
  C2|छथ                0.3092
  W1|नहि               0.3029
  C3|लिख               0.3029
  C3|िलि               0.2965
  C3|थिन               0.2815
  C3|नहि               0.2788
  C3|न्ह               0.2536
  C3|हिल               0.2379
  C2|िख                0.2307
  C3|िन्               0.2273
--- end example ---


Training custom softmax logistic regression...
epoch   1/20  train_loss=2.7907  val_acc=0.8787
epoch   2/20  train_loss=2.1735  val_acc=0.9